In [ ]:
# input Ui, Uj, Aij, Vi and g
# CHANGED TO Roe-averaged 
# output: roe-averaged matrix dR(tildeU)/tildeU
# directly use formula from chungs book
# return numerical values
# tildeU = roe-averaged of Ui and Uj
# roe-averaged matrix = (a1 * n1 + a2 * n2 + a3 * n3)* area = nx * a1 + ny * a2 + nz *a3

from sympy import symbols, Matrix, lambdify
import numpy as np

# Flow state variables (Roe-averaged quantities)
# create symbols 
u, v, w, E, g = symbols('u v w E gamma')

# area normal vectors
nx, ny, nz = symbols('nx ny nz') 

# Helper terms
q2 = u**2 + v**2 + w**2  # u norm
g1 = g - 1
g3 = g - 3

# a1, a2, a3 from Chung, T. J., "Computational Fluid Dynamics" Appendix A
# roe matrix = nx * a1 + ny * a2 + nz *a3 
# area normal vector = [nx,ny,nz]

a1 = Matrix([
    [0, 1, 0, 0, 0],
    [(g3/2)*u**2 + (g1/2)*(v**2 + w**2), (3-g)*u, (1-g)*v, (1-g)*w, g1],
    [-u*v, v, u, 0, 0],
    [-u*w, w, 0, u, 0],
    [-g*E*u + g1*u*q2, g*E + (1-g)/2*(3*u**2 + v**2 + w**2), (1-g)*u*v, (1-g)*u*w, g*u]
])

a2 = Matrix([
    [0, 0, 1, 0, 0],
    [-u*v, v, u, 0, 0],
    [(g3/2)*v**2 + (g1/2)*(u**2 + w**2), (1-g)*u, (3-g)*v, (1-g)*w, g1],
    [-v*w, 0, w, v, 0],
    [-g*E*v + g1*v*q2, (1-g)*u*v, g*E + (1-g)/2*(u**2 + 3*v**2 + w**2), (1-g)*v*w, g*v]
])

a3 = Matrix([
    [0, 0, 0, 1, 0],
    [-u*w, w, 0, u, 0],
    [-v*w, 0, w, v, 0],
    [(g3/2)*w**2 + (g1/2)*(u**2 + v**2), (1-g)*u, (1-g)*v, (3-g)*w, g1],
    [-g*E*w + g1*w*q2, (1-g)*u*w, (1-g)*v*w, g*E + (1-g)/2*(u**2 + v**2 + 3*w**2), g*w]
])

# 4. Construct the Jacobian using pure scalar area components
roe_averaged_jacobian = a1 * nx + a2 * ny + a3 * nz

# 5. Lambdify! (Guaranteed to work because no MatrixElements exist here)
roe_matrix = lambdify(
    (u, v, w, E, g, nx, ny, nz), 
    roe_averaged_jacobian, 
    modules='numpy'
)

In [11]:
if __name__ == "__main__":
    # Test Case: Simple X-direction flow, Normal vector pointing in +X
    # Roe-averaged velocity: u=1.0, v=0, w=0
    # Roe-averaged Energy: $E=2.5$
    # gamma = 1.4
    # area normal vector = [1,0,0]

    # known results:
    # [[ 0.   1.   0.   0.   0. ]
    # [-0.8  1.6  0.   0.   0.4]
    # [ 0.   0.   1.   0.   0. ]
    # [ 0.   0.   0.   1.   0. ]
    # [-3.1  2.9  0.   0.   1.4]]

    
    # result = roe_matrix(u, v, w, E, gamma, nx, ny, nz)
    res = roe_matrix(1.0, 0.0, 0.0, 2.5, 1.4, 1.0, 0.0, 0.0)
    
    print("Verification: Roe-Averaged Jacobian Matrix (X-dir flow)")
    print(res)
    assert res[0,1] == 1.0, "Error: Continuity derivative should be 1.0"

Verification: Roe-Averaged Jacobian Matrix (X-dir flow)
[[ 0.   1.   0.   0.   0. ]
 [-0.8  1.6  0.   0.   0.4]
 [ 0.   0.   1.   0.   0. ]
 [ 0.   0.   0.   1.   0. ]
 [-3.1  2.9  0.   0.   1.4]]
